# Milestone 1: Data Exploration and Preprocessing

This notebook prepares Amazon review data for retrieval.

Scope:
- Load dataset from repo structure
- Inspect data
- Select fields
- Clean and tokenize text
- Export clean dataset as parquet


In [1]:
from pathlib import Path
import pandas as pd
from typing import Dict

def find_repo_root(max_up: int = 6) -> Path:
    p = Path.cwd()
    for _ in range(max_up):
        if (p / 'README.md').exists() or (p / '.git').exists():
            return p
        p = p.parent
    return Path.cwd()

repo_root = find_repo_root()
print('Repo root:', repo_root)

sample_dir = repo_root / 'data' / 'processed'
samples = {
    'All_Beauty': sample_dir / 'sample_All_Beauty.jsonl',
    'Health_and_Personal_Care': sample_dir / 'sample_Health_and_Personal_Care.jsonl'
}

dfs: Dict[str, pd.DataFrame] = {}
for name, p in samples.items():
    if not p.exists():
        print(f'File missing: {p} -- skipping {name}')
        dfs[name] = pd.DataFrame()
        continue
    try:
        df = pd.read_json(p, lines=True)
    except Exception as e:
        print(f'Error reading {p}:', e)
        dfs[name] = pd.DataFrame()
        continue
    
    dfs[name] = df.copy()
    print(f'Loaded {name}: rows={len(df)}, cols={len(df.columns)}')

_dfs = dfs

Repo root: c:\Users\Omowunmi\mds_labs\DSCI_575_project_omo001_deepray
Loaded All_Beauty: rows=200, cols=10
Loaded Health_and_Personal_Care: rows=200, cols=10


In [ ]:
for name, df in _dfs.items():
    if df.empty:
        continue
    print(f"\n{name} sample record:")
    print(df.iloc[0].to_dict())


All_Beauty sample record:
{'rating': 5, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': Timestamp('2020-05-05 14:08:48.923000'), 'helpful_vote': 0, 'verified_purchase': True}

Health_and_Personal_Care sample record:
{'rating': 4, 'title': '12 mg is 12 on the periodic table people! Mg for magnesium', 'text': 'This review is more to clarify someone else’s review bc they didn’t understand understand the labeling!  It shows 1000mg as advertised & another little label says 12mg bc 12 is on the periodic table for magnesium!  I realize not everyone takes chemis

## Field Selection

Fields used:
- doc_id from asin
- title
- text
- rating

These fields support keyword retrieval and ranking.

In [3]:
def select_fields(df):
    return pd.DataFrame({
        'doc_id': df.get('asin'),
        'title': df.get('title', ''),
        'text': df.get('text', ''),
        'rating': df.get('rating', 0)
    })

clean_dfs = {}
for name, df in _dfs.items():
    if df.empty:
        continue
    clean_dfs[name] = select_fields(df)

clean_dfs[list(clean_dfs.keys())[0]].head()

,doc_id,title,text,rating
0,B00YQ6X8EO,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,5
1,B081TJ8YS3,Works great but smells a little weird.,"This product does what I need it to do, I just...",4
2,B07PNNCSP9,Yes!,"Smells good, feels great!",5
3,B09JS339BZ,Synthetic feeling,Felt synthetic,1
4,B08BZ63GMJ,A+,Love it,5


## Text Preprocessing

Steps:
- Lowercase text
- Remove noisy characters
- Normalize whitespace
- Combine title and review text


In [4]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s\.,!?\'\"-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
for name, df in clean_dfs.items():
    df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
    df['clean_text'] = df['combined'].apply(clean_text)

clean_dfs[list(clean_dfs.keys())[0]][['doc_id', 'clean_text']].head()

,doc_id,clean_text
0,B00YQ6X8EO,such a lovely scent but not overpowering. this...
1,B081TJ8YS3,works great but smells a little weird. this pr...
2,B07PNNCSP9,"yes! smells good, feels great!"
3,B09JS339BZ,synthetic feeling felt synthetic
4,B08BZ63GMJ,a love it


## Export Clean Dataset

Save processed data as parquet for reuse in retrieval tasks.

In [6]:
output_dir = repo_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in clean_dfs.items():
    out_path = output_dir / f"{name}_clean.parquet"
    df_to_save = df[['doc_id', 'title', 'clean_text', 'rating']].copy()
    df_to_save.to_parquet(out_path, index=False)
    print('Saved:', out_path)

Saved: c:\Users\Omowunmi\mds_labs\DSCI_575_project_omo001_deepray\data\processed\All_Beauty_clean.parquet
Saved: c:\Users\Omowunmi\mds_labs\DSCI_575_project_omo001_deepray\data\processed\Health_and_Personal_Care_clean.parquet
